In [15]:
from GradientGang.Pipeline.DataLoader.DataLoader import DataModule

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
# First, check if the files exist and what's in them
import pandas as pd
import os

data_dir = "../../dataset/PirateProcessed/"
train_file = os.path.join(data_dir, "pirate_pain_train.csv")
global_file = os.path.join(data_dir, "train_global_features.csv")

print(f"Checking files:")
print(f"  Train file exists: {os.path.exists(train_file)}")
print(f"  Global features file exists: {os.path.exists(global_file)}")

if os.path.exists(train_file):
    train_df = pd.read_csv(train_file)
    print(f"\nTrain data shape: {train_df.shape}")
    print(f"Train data columns ({len(train_df.columns)}): {train_df.columns.tolist()[:10]}...")
    print(f"Train data dtypes:\n{train_df.dtypes}")
    
if os.path.exists(global_file):
    global_df = pd.read_csv(global_file)
    print(f"\nGlobal features shape: {global_df.shape}")
    print(f"Global features columns ({len(global_df.columns)}): {global_df.columns.tolist()[:10]}...")


Checking files:
  Train file exists: True
  Global features file exists: True

Train data shape: (105760, 36)
Train data columns (36): ['sample_index', 'time', 'pain_survey_1', 'pain_survey_2', 'pain_survey_3', 'pain_survey_4', 'joint_00', 'joint_01', 'joint_02', 'joint_03']...
Train data dtypes:
sample_index       int64
time               int64
pain_survey_1    float64
pain_survey_2    float64
pain_survey_3    float64
pain_survey_4    float64
joint_00         float64
joint_01         float64
joint_02         float64
joint_03         float64
joint_04         float64
joint_05         float64
joint_06         float64
joint_07         float64
joint_08         float64
joint_09         float64
joint_10         float64
joint_11         float64
joint_12         float64
joint_13         float64
joint_14         float64
joint_15         float64
joint_16         float64
joint_17         float64
joint_18         float64
joint_19         float64
joint_20         float64
joint_21         float64
jo

In [17]:
params = {
    'data_dir': "../../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'train_global_features_file': "train_global_features.csv",
    'test_global_features_file': "test_global_features.csv",
    'batch_size': 32,
    'num_workers': 0,
    'val_split': 0.1,
    'shuffle': True,
}

dataLoader = DataModule(params=params)
dataLoader.setup(stage='fit')

trainLoader = dataLoader.train_dataloader()
valLoader = dataLoader.val_dataloader()

dataLoader.setup(stage='test')
testLoader = dataLoader.test_dataloader()

# Print dataset info
datasetInfo = dataLoader.getDatasetInfo()
print(f"\n📊 Dataset Information:")
print(f"  - Time series shape: {datasetInfo['timeSeriesShape']}")
print(f"  - Global features shape: {datasetInfo['globalFeaturesShape']}")
print(f"  - Number of classes: {datasetInfo['numClasses']}")



📊 Dataset Information:
  - Time series shape: torch.Size([34, 160])
  - Global features shape: torch.Size([32])
  - Number of classes: 3


### 🔧 Fixed: DataLoader now properly loads global features from separate CSV

The issue was that after loading global features from the separate CSV file, the code was overwriting them by trying to extract from the time series CSV (which doesn't have those columns). Now it correctly preserves the loaded global features.

In [19]:
for batch in trainLoader:
    (time_series, global_features), labels = batch
    print("Train Batch - Time Series Shape:", time_series.shape)
    print("Train Batch - Global Features Shape:", global_features.shape)
    print("Train Batch - Labels Shape:", labels.shape)
    break

Train Batch - Time Series Shape: torch.Size([32, 34, 160])
Train Batch - Global Features Shape: torch.Size([32, 32])
Train Batch - Labels Shape: torch.Size([32])


## 🔄 K-Fold Cross-Validation Test

Testing the new K-Fold functionality integrated into DataModule.

In [22]:
# Test K-Fold Cross-Validation
kfold_params = {
    'data_dir': "../../dataset/PirateProcessed/",
    'train_file_name': "pirate_pain_train.csv",
    'train_file_name_labels': "pirate_pain_train_labels.csv",
    'test_file_name': "pirate_pain_test.csv",
    'train_global_features_file': "train_global_features.csv",
    'test_global_features_file': "test_global_features.csv",
    'batch_size': 32,
    'num_workers': 0,
    'use_kfold': True,  # Enable K-Fold
    'n_folds': 5,        # 5-fold CV
    'val_split': 0.1,
}

kfold_dataLoader = DataModule(params=kfold_params)
kfold_dataLoader.setup(stage='fit')

print("📊 K-Fold Configuration:")
fold_info = kfold_dataLoader.get_fold_info()
for key, value in fold_info.items():
    print(f"  {key}: {value}")

# Test each fold
print("\n🔄 Testing each fold:")
for fold in range(5):
    kfold_dataLoader.setup_fold(fold, include_test_in_train=True)
    
    train_loader = kfold_dataLoader.train_dataloader()
    val_loader = kfold_dataLoader.val_dataloader()
    
    # Get one batch from each
    train_batch = next(iter(train_loader))
    val_batch = next(iter(val_loader))
    
    (train_ts, train_gf), train_labels = train_batch
    (val_ts, val_gf), val_labels = val_batch
    
    fold_info = kfold_dataLoader.get_fold_info()
    print(f"\n  Fold {fold}:")
    print(f"    Train samples: {fold_info['train_size']}, Val samples: {fold_info['val_size']}")
    print(f"    Train batch - TS: {train_ts.shape}, GF: {train_gf.shape}, Labels: {train_labels.shape}")
    print(f"    Val batch - TS: {val_ts.shape}, GF: {val_gf.shape}, Labels: {val_labels.shape}")

print("\n✅ K-Fold Cross-Validation is working!")

📊 K-Fold Configuration:
  use_kfold: True
  n_folds: 5
  current_fold: None
  full_dataset_size: 661
  train_size: 595
  val_size: 66

🔄 Testing each fold:

  Fold 0:
    Train samples: 528, Val samples: 133
    Train batch - TS: torch.Size([32, 34, 160]), GF: torch.Size([32, 32]), Labels: torch.Size([32])
    Val batch - TS: torch.Size([32, 34, 160]), GF: torch.Size([32, 32]), Labels: torch.Size([32])

  Fold 1:
    Train samples: 529, Val samples: 132
    Train batch - TS: torch.Size([32, 34, 160]), GF: torch.Size([32, 32]), Labels: torch.Size([32])
    Val batch - TS: torch.Size([32, 34, 160]), GF: torch.Size([32, 32]), Labels: torch.Size([32])

  Fold 2:
    Train samples: 529, Val samples: 132
    Train batch - TS: torch.Size([32, 34, 160]), GF: torch.Size([32, 32]), Labels: torch.Size([32])
    Val batch - TS: torch.Size([32, 34, 160]), GF: torch.Size([32, 32]), Labels: torch.Size([32])

  Fold 3:
    Train samples: 529, Val samples: 132
    Train batch - TS: torch.Size([32, 34, 